# NLP Lab Task 04 — Complete Solutions
**Notebook guide:** Run cells from top to bottom. Some cells download NLTK resources or the spaCy English model on first use. Internet access may be required for those downloads.

## Setup

In [ ]:
import re, time, string, html
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import nltk

for resource in ["stopwords", "punkt", "punkt_tab", "movie_reviews", "words", "cmudict"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"Could not download {resource}: {e}")

print("Python/NLTK setup ready")

## Task 1 — Lowercasing Basics and Its Drawbacks

In [ ]:
corpus = [
    "Apple released a new phone yesterday.",
    "I ate an apple after lunch.",
    "APPLE is written in capital letters on the poster.",
    "The US announced a new policy.",
    "Please send the report to us by noon.",
    "The company works with Apple in California.",
    "She likes apple pie more than cake.",
    "US markets reacted to the news.",
    "The word us appears in this sentence.",
    "I bought an APPLE from the store.",
    "May is a month, and may can express possibility.",
    "Turkey is a country; the turkey was roasted."
]
tokens_before = [w for sent in corpus for w in re.findall(r"\b\w+\b", sent)]
tokens_after = [w.lower() for w in tokens_before]
vocab_before = set(tokens_before)
vocab_after = set(tokens_after)
print("Vocabulary size before lowercasing:", len(vocab_before))
print("Vocabulary size after lowercasing: ", len(vocab_after))
print("Vocabulary reduction:", len(vocab_before) - len(vocab_after))
print("\\nCase-sensitive forms merged:")
for word in sorted(vocab_before):
    forms = [x for x in vocab_before if x.lower() == word.lower()]
    if len(forms) > 1 and word == min(forms):
        print(forms, "->", word.lower())

In [ ]:
ambiguities = [
    ('Apple released a new phone.', 'I ate an apple.', 'Apple can mean the company or the fruit.'),
    ('The US announced a policy.', 'Send the report to us.', 'US is the country abbreviation; us is a pronoun.'),
    ('Turkey joined the meeting.', 'We ate turkey for dinner.', 'Turkey can name a country or a bird/food.')
]
for a, b, why in ambiguities:
    print(f"• {a} / {b}\n  Ambiguity after lowercasing: {why}\n")

**Discussion:** Lowercasing reduces vocabulary sparsity and can improve matching in search and text classification when capitalization is incidental. However, it can harm **named-entity recognition (NER)** by erasing capitalization cues (e.g., *Apple* as an organization), **sentiment analysis** when capitalization signals emphasis (e.g., “I LOVE it”), and **search** when users need case-sensitive distinctions such as acronyms, product codes, or names. A practical system may preserve the original text and use a lowercased copy only where normalization is helpful.

## Task 2 — Noise Removal Pipeline

In [ ]:
noisy = [
    "Loved this phone!!! 😍 Visit https://shop.example.com @brand #great",
    "<p>Terrible service...</p>  never coming back 😡",
    "I paid $100 for C++ lessons!!!   #coding",
    "@support Please help me with my order: https://x.co/a",
    "<b>Excellent</b> quality 😊😊 #happy",
    "This movie was NOT good!!! http://review.example",
    "Shipping was fast   and   packaging was neat.",
    "I hate bugs in C++ code. #programming",
    "<div>Five stars!</div> @seller",
    "Not worth it...  not at all 😤",
    "The price is $25.50, which is fair.",
    "Check this out: https://example.org/path?q=1 #wow",
    "Customer service replied <i>quickly</i>.",
    "Wow!!!     absolutely amazing ❤️❤️",
    "I can't recommend it. @friend #honest-review"
]

def remove_urls(text):
    return re.sub(r"https?://\S+|www\.\S+", " ", text)

def remove_html_tags(text):
    return re.sub(r"<[^>]+>", " ", html.unescape(text))

def handle_mentions_hashtags(text, mode="remove"):
    # mode='remove' deletes @mentions and the # symbol while preserving hashtag words.
    text = re.sub(r"@\w+", " ", text)
    if mode == "remove":
        text = re.sub(r"#(\w+)", r"\1", text)
    elif mode == "delete":
        text = re.sub(r"#\w+", " ", text)
    return text

def remove_punctuation(text, preserve_symbols=True):
    # Conservative mode preserves C++, apostrophes, decimal points, and currency signs.
    if preserve_symbols:
        text = re.sub(r"[!?.,;:()\[\]{}\"“”‘’]", " ", text)
    else:
        text = re.sub(r"[^\w\s]", " ", text)
    return text

def remove_extra_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

def clean_text(text, preserve_symbols=True, hashtag_mode="remove"):
    text = remove_urls(text)
    text = remove_html_tags(text)
    text = handle_mentions_hashtags(text, mode=hashtag_mode)
    # Strip emojis/symbols outside basic word characters, while retaining selected useful symbols.
    if preserve_symbols:
        text = re.sub(r"[^\w\s$+.']", " ", text)
    else:
        text = re.sub(r"[^\w\s]", " ", text)
    text = remove_punctuation(text, preserve_symbols=preserve_symbols)
    return remove_extra_whitespace(text)

examples = pd.DataFrame({
    "Before": noisy[:5],
    "After (conservative)": [clean_text(x) for x in noisy[:5]],
    "After (aggressive)": [clean_text(x, preserve_symbols=False, hashtag_mode="delete") for x in noisy[:5]]
})
display(examples)

In [ ]:
sample = "Look at https://example.com #NLP @alice <b>great!!!</b>"
order_a = remove_extra_whitespace(remove_html_tags(remove_urls(sample)))
order_b = remove_urls(remove_extra_whitespace(remove_html_tags(sample)))
print("Order A:", order_a)
print("Order B:", order_b)
print("Same output here?", order_a == order_b)
print("Order can matter when an earlier regex creates/removes boundaries that a later regex relies on.")
print("Over-cleaning example:", clean_text("I paid $100 for C++!", preserve_symbols=False))
print("Conservative example:", clean_text("I paid $100 for C++!", preserve_symbols=True))

**Over-cleaning note:** Removing all punctuation can damage programming terms such as `C++`, contractions such as `can't`, decimals such as `$25.50`, and prices such as `$100`. URL removal can also erase useful domain information. Choose cleaning rules for the downstream task rather than treating every symbol as noise.

## Task 3 — Stop-Word Removal and Its Effect on Vocabulary

In [ ]:
from nltk.corpus import stopwords, movie_reviews
from nltk.tokenize import word_tokenize, sent_tokenize

stop_words = set(stopwords.words("english"))
print("First 20 NLTK English stop words:", sorted(stop_words)[:20])
print("Total stop-word count:", len(stop_words))

review_ids = movie_reviews.fileids()[:300]
reviews = [movie_reviews.raw(fid)[:1800] for fid in review_ids]
def tokenize(text):
    try:
        return [t.lower() for t in word_tokenize(text) if re.search(r"[A-Za-z]", t)]
    except LookupError:
        return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text.lower())

before_tokens = [tokenize(x) for x in reviews]
after_tokens = [[t for t in toks if t not in stop_words] for toks in before_tokens]
def stats(token_lists):
    flat = [t for doc in token_lists for t in doc]
    lengths = [len(doc) for doc in token_lists]
    return {"Token count": len(flat), "Vocabulary size": len(set(flat)),
            "Average tokens/document": round(sum(lengths)/len(lengths), 2)}
display(pd.DataFrame([stats(before_tokens), stats(after_tokens)],
                     index=["Before removal", "After removal"]))

In [ ]:
def plot_top15(token_lists, title):
    counts = Counter(t for doc in token_lists for t in doc).most_common(15)
    words, freqs = zip(*counts)
    plt.figure(figsize=(10, 5))
    plt.bar(words, freqs)
    plt.xticks(rotation=45, ha="right")
    plt.title(title)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

plot_top15(before_tokens, "Top 15 words before stop-word removal")
plot_top15(after_tokens, "Top 15 words after stop-word removal")

In [ ]:
# Custom list: additions target a movie-review corpus; removals protect meaning/sentiment.
custom_stop_words = set(stop_words)
domain_additions = {"film", "movie", "cinema", "actor", "actress", "director", "scene", "plot", "character", "story"}
custom_stop_words.update(domain_additions)
protected_words = {"no", "nor", "not", "never", "very", "against", "but", "without", "shouldn't", "wouldn't"}
custom_stop_words.difference_update(protected_words)
print("Added domain words (10):", sorted(domain_additions))
print("Why add them? In broad movie-review topic modeling, these frequent domain terms may dominate topic features.")
print("Removed/protected words:", sorted(protected_words))
print("Why protect them? Negation, contrast, intensity, and modal meaning can change sentiment or intent.")

## Task 4 — Risks of Stop-Word Removal (Sentiment, MT, Chatbots)

In [ ]:
negation_examples = [
    "This is not good.",
    "I am never happy with the service.",
    "There is no problem with the update.",
    "The food was not bad.",
    "I do not recommend this product."
]
def remove_stops(text, stops=stop_words):
    return " ".join(t for t in tokenize(text) if t not in stops)
for s in negation_examples:
    print("Original:", s)
    print("Blind removal:", remove_stops(s))
    print()

In [ ]:
# Three examples with illustrative Urdu renderings. For a formal submission, verify with Google Translate
# or another translator and record the translator's actual output.
mt_examples = [
    ("I do not want to cancel my order.", "میں اپنا آرڈر منسوخ نہیں کرنا چاہتا/چاہتی۔"),
    ("The package was delivered to my house.", "پارسل میرے گھر پہنچا دیا گیا۔"),
    ("She is not unhappy with the result.", "وہ نتیجے سے ناخوش نہیں ہے۔")
]
for en, ur in mt_examples:
    reduced = remove_stops(en)
    print("Original English:", en)
    print("Reduced English:", reduced)
    print("Illustrative Urdu of original:", ur)
    print("Risk: translating the reduced text may omit negation, articles, prepositions, or tense cues.\\n")

In [ ]:
chatbot_queries = [
    "How do I not get charged?",
    "What is the status of my order?",
    "Can you cancel my subscription?",
    "I do not want a refund.",
    "Where has my package not been delivered?"
]
for q in chatbot_queries:
    print(f"Query: {q}\\nAfter blind stop-word removal: {remove_stops(q)}\\n")

In [ ]:
safe_stops = set(stop_words) - {"no", "nor", "not", "never", "without", "against", "but", "cannot", "don't", "doesn't", "didn't", "isn't", "wasn't", "won't", "wouldn't"}
print("Safe strategy: preserve negation and contrast words; consider preserving function words for translation and intent classification.")
for s in negation_examples:
    print("Original:", s)
    print("Blind:", remove_stops(s))
    print("Safe:", remove_stops(s, safe_stops), "\\n")

## Task 5 — Hands-On with the Porter Stemmer

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer
porter = PorterStemmer()
snowball = SnowballStemmer("english")
words = [
    "running","runs","runner","studies","studying","happily","generalization","generously",
    "better","best","was","are","children","ponies","relational","conditional","digitizer",
    "vietnamization","triplicate","formalize","electricity","hopeful","hopeless","connection",
    "connected","connections","organization","organize","university","universe","policy",
    "police","cats","dogs","agreed","agreement","relate","relativity","sky","lying"
]
def rough_correctness(w, stem):
    # Heuristic annotation, not a gold linguistic judgement: stemmers return search keys, not necessarily words.
    if w in {"was","are","better","best","children"} and stem not in {"be","good","child"}:
        return "Incorrect / not lemma"
    if stem == w.lower() or stem in w.lower():
        return "Reasonable stem"
    return "Check manually"
stem_table = pd.DataFrame({
    "Word": words,
    "Porter stem": [porter.stem(w) for w in words],
    "Correct/Incorrect (heuristic)": [rough_correctness(w, porter.stem(w)) for w in words],
    "Snowball stem": [snowball.stem(w) for w in words]
})
display(stem_table)
print("Words where Porter and Snowball differ:")
display(stem_table[stem_table["Porter stem"] != stem_table["Snowball stem"]][["Word","Porter stem","Snowball stem"]])

**Over-stemming examples (collisions that may be undesirable):**
- `university` and `universe` may collapse to similar/identical stems under aggressive suffix rules, despite different meanings.
- `policy` and `police` can be reduced to similar short forms by some stemming algorithms.
- `organization` and `organ` may share a stem-like prefix even though their meanings differ.

**Under-stemming examples (related forms not always reduced to the same useful base):**
- `good` and `better` remain different (`good` vs. `better`), because Porter is not a semantic lemmatizer.
- `was` and `be` remain different, despite being forms of the same verb.
- `children` and `child` remain different, because Porter does not generally perform irregular morphology.

Exact collision behavior can vary by stemmer/version; inspect the table produced above and report the actual outputs from your installed NLTK version.

## Task 6 — Lemmatization (spaCy) vs. Stemming

In [ ]:
paragraph = (
    "The researchers were studying several complex linguistic problems while the students wrote reports. "
    "A better model can identify words that were running through long sentences and normalize their forms. "
    "People have gone to different cities, and the children were happier after they had eaten. "
    "The system processes documents, analyzes meanings, and compares the results with earlier experiments. "
    "Carefully written explanations are more useful than simply deleting frequent words from every sentence. "
    "A company was organizing meetings, but the employees did not agree about the proposed changes. "
    "These observations suggest that context and grammatical role matter when interpreting language. "
    "The quick brown foxes jumped over the lazy dogs, while a researcher was better prepared for the task. "
    "They have been making decisions and were choosing the most appropriate tools for the analysis. "
    "Natural language processing combines statistics, linguistics, and software engineering to extract useful information."
)
print("Word count:", len(re.findall(r"\b\w+\b", paragraph)))
try:
    import spacy
    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError:
        print("Downloading spaCy model...")
        from spacy.cli import download
        download("en_core_web_sm")
        nlp = spacy.load("en_core_web_sm")
    doc = nlp(paragraph)
    rows = []
    for tok in doc:
        if tok.is_alpha:
            rows.append({"Token": tok.text, "POS": tok.pos_, "Stem": porter.stem(tok.text), "Lemma": tok.lemma_})
    comparison = pd.DataFrame(rows)
    display(comparison.head(60))
except Exception as e:
    print("spaCy/model unavailable:", e)
    print("Install with: pip install spacy && python -m spacy download en_core_web_sm")

In [ ]:
# Speed comparison on the same paragraph, repeated for a stable rough timing.
tokens_for_speed = [t for t in re.findall(r"\b[A-Za-z]+\b", paragraph)]
start = time.perf_counter()
for _ in range(500):
    [porter.stem(t) for t in tokens_for_speed]
porter_seconds = time.perf_counter() - start
try:
    start = time.perf_counter()
    for _ in range(500):
        _ = nlp(paragraph)
    spacy_seconds = time.perf_counter() - start
    print(f"Porter stemming: {porter_seconds:.4f} sec / 500 runs")
    print(f"spaCy pipeline:  {spacy_seconds:.4f} sec / 500 runs")
except Exception as e:
    print("spaCy timing unavailable:", e)

**Interpretation:** Lemmatization uses vocabulary and grammatical context; for example, spaCy may map `was → be`, `better → well/good` depending on POS/context, and `children → child`. Porter produces short stems that are not necessarily words. The POS tag helps lemmatization select the appropriate base form (e.g., noun vs. verb). Choose stemming for fast, rough search/indexing; choose lemmatization when readable normalized forms or grammatical accuracy matter. Timings depend on hardware, model loading, and document size.

## Task 7 — Preprocessing Pipeline and Classification Experiment

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# 60 reproducible synthetic documents, two balanced classes.
positive_templates = [
    "The product is excellent and works perfectly",
    "I love this wonderful service and recommend it",
    "Amazing quality and fast delivery made me happy",
    "The experience was great and the support was helpful",
    "Very satisfied with this reliable and useful product"
]
negative_templates = [
    "The product is terrible and never works properly",
    "I hate this awful service and cannot recommend it",
    "Poor quality and slow delivery made me unhappy",
    "The experience was bad and the support was unhelpful",
    "Very disappointed with this unreliable and useless product"
]
docs, labels = [], []
for i in range(30):
    docs.append(positive_templates[i % len(positive_templates)] + f" purchase reference item{i%7}.")
    labels.append("positive")
    docs.append(negative_templates[i % len(negative_templates)] + f" complaint reference item{i%7}.")
    labels.append("negative")

def preprocess_doc(text, config):
    if config.get("lower", True):
        text = text.lower()
    if config.get("noise", True):
        text = remove_urls(text)
        text = remove_html_tags(text)
        text = re.sub(r"@\w+|#(\\w+)", r"\\1", text)
        text = remove_extra_whitespace(text)
    toks = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|\\$\\d+(?:\\.\\d+)?|\\w+\\+\\+", text)
    if config.get("stop", False):
        toks = [t for t in toks if t.lower() not in safe_stops]
    if config.get("stem", False):
        toks = [porter.stem(t) for t in toks]
    if config.get("lemma", False):
        try:
            toks = [t.lemma_ for t in nlp(" ".join(toks))]
        except Exception:
            pass
    return " ".join(toks)

configs = {
    "Lowercase only": {"lower": True, "noise": False, "stop": False},
    "Lowercase + stop words": {"lower": True, "noise": False, "stop": True},
    "Lowercase + stemming": {"lower": True, "noise": False, "stop": False, "stem": True},
    "Lowercase + lemmatization": {"lower": True, "noise": False, "stop": False, "lemma": True},
}
X_train_idx, X_test_idx = train_test_split(range(len(docs)), test_size=0.25, random_state=42, stratify=labels)
results = []
for name, cfg in configs.items():
    processed = [preprocess_doc(d, cfg) for d in docs]
    vocab_size = len(set(tok for d in processed for tok in d.split()))
    avg_tokens = sum(len(d.split()) for d in processed) / len(processed)
    train_text = [processed[i] for i in X_train_idx]
    test_text = [processed[i] for i in X_test_idx]
    y_train = [labels[i] for i in X_train_idx]
    y_test = [labels[i] for i in X_test_idx]
    vec = TfidfVectorizer()
    Xtr = vec.fit_transform(train_text)
    Xte = vec.transform(test_text)
    clf = MultinomialNB().fit(Xtr, y_train)
    pred = clf.predict(Xte)
    results.append({"Configuration": name, "Vocabulary size": vocab_size,
                    "Average tokens/document": round(avg_tokens, 2),
                    "Accuracy": round(accuracy_score(y_test, pred), 3)})
results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
display(results_df)

**Analysis (interpret the measured table above):**
1. Lowercasing merges capitalization variants and generally reduces redundant features.
2. Stop-word removal reduces document length, but can remove useful grammatical or sentiment cues if the list is unsafe.
3. Stemming merges inflectional variants and may help when training data is small, though it can create misleading collisions.
4. Lemmatization often preserves more interpretable base forms but costs more runtime and depends on POS/context.
5. The best accuracy is the configuration with the highest held-out score in the table; this synthetic dataset is intentionally simple, so results may be tied or unusually high.
6. Re-run with a real, larger labeled corpus for a meaningful generalization claim.

## Task 8 — Minimum Edit Distance (Levenshtein)

In [ ]:
def levenshtein_recursive(a, b):
    # Exponential-time reference implementation: intentionally no memoization.
    if not a: return len(b)
    if not b: return len(a)
    cost = 0 if a[-1] == b[-1] else 1
    return min(
        levenshtein_recursive(a[:-1], b) + 1,      # deletion
        levenshtein_recursive(a, b[:-1]) + 1,      # insertion
        levenshtein_recursive(a[:-1], b[:-1]) + cost # substitution/match
    )

def levenshtein_dp(a, b, substitution_cost=1, show_table=False):
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            sub = 0 if a[i-1] == b[j-1] else substitution_cost
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+sub)
    if show_table:
        display(pd.DataFrame(dp, index=["∅"]+list(a), columns=["∅"]+list(b)))
    return dp[m][n], dp

pairs = [("intention","execution"), ("kitten","sitting"), ("sunday","saturday")]
for a, b in pairs:
    dist, dp = levenshtein_dp(a, b, show_table=True)
    print(f"{a!r} -> {b!r}: distance={dist}")
print("Recursive kitten/sitting:", levenshtein_recursive("kitten", "sitting"))
print("Limitation: recursive calls repeat the same subproblems, making runtime exponential for longer strings.")

In [ ]:
def backtrace(a, b, dp, substitution_cost=1):
    i, j = len(a), len(b)
    ops = []
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            sub = 0 if a[i-1] == b[j-1] else substitution_cost
            if dp[i][j] == dp[i-1][j-1] + sub:
                if sub == 0:
                    ops.append(("MATCH", a[i-1], b[j-1]))
                else:
                    ops.append(("SUBSTITUTE", a[i-1], b[j-1], sub))
                i -= 1; j -= 1
                continue
        if i > 0 and dp[i][j] == dp[i-1][j] + 1:
            ops.append(("DELETE", a[i-1]))
            i -= 1
        elif j > 0:
            ops.append(("INSERT", b[j-1]))
            j -= 1
        else:
            break
    return list(reversed(ops))

for a, b in pairs:
    d1, table1 = levenshtein_dp(a, b, substitution_cost=1)
    d2, table2 = levenshtein_dp(a, b, substitution_cost=2)
    print(f"\\n{a} -> {b}")
    print("Substitution cost 1:", d1, backtrace(a,b,table1,1))
    print("Substitution cost 2:", d2, backtrace(a,b,table2,2))

**By-hand DP tables:** The displayed tables are the dynamic-programming matrices for each requested pair; rows are prefixes of the first string and columns are prefixes of the second. Each cell is the minimum cost to transform one prefix into the other. Substitution cost 2 can increase the optimal distance and may favor a delete-plus-insert path over a substitution.

## Task 9 — Spelling Correction with Edit Distance

In [ ]:
from nltk.corpus import words as nltk_words, cmudict
try:
    dictionary = set(w.lower() for w in nltk_words.words() if w.isalpha() and len(w) > 1)
except Exception:
    dictionary = set()
if len(dictionary) < 5000:
    try:
        dictionary |= set(w.lower() for w in cmudict.words() if w.isalpha() and len(w) > 1)
    except Exception:
        pass
print("Dictionary size:", len(dictionary))
if len(dictionary) < 5000:
    print("Dictionary corpus has fewer than 5,000 entries. Run nltk.download('words') or nltk.download('cmudict') and retry.")

In [ ]:
def edit_distance(a, b):
    # Efficient rolling-row Levenshtein distance, unit costs.
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(cur[-1]+1, prev[j]+1, prev[j-1]+(ca != cb)))
        prev = cur
    return prev[-1]

# Frequency from CMU dictionary entry counts is a usable frequency proxy, not a true corpus frequency.
try:
    cmu_counts = Counter(w.lower() for w in cmudict.words() if w.isalpha())
except Exception:
    cmu_counts = Counter()
def suggest(word, k=5, use_frequency=False):
    word = word.lower()
    scored = [(edit_distance(word, cand), cand) for cand in dictionary]
    if use_frequency:
        scored.sort(key=lambda x: (x[0], -cmu_counts.get(x[1], 0), x[1]))
    else:
        scored.sort(key=lambda x: (x[0], x[1]))
    return scored[:k]

tests = {
    "recieve": "receive",
    "definately": "definitely",
    "acommodate": "accommodate",
    "seperate": "separate",
    "occured": "occurred",
    "enviroment": "environment",
    "adress": "address",
    "beleive": "believe",
    "freind": "friend",
    "goverment": "government",
    "untill": "until",
    "wierd": "weird",
    "thier": "their",
    "begining": "beginning",
    "tommorow": "tomorrow"
}
rows = []
for misspelling, gold in tests.items():
    plain = suggest(misspelling, k=5, use_frequency=False)
    freq = suggest(misspelling, k=5, use_frequency=True)
    rows.append({
        "Misspelling": misspelling, "Expected": gold,
        "Top-1 (distance only)": plain[0][1],
        "Top-1 (frequency tie-break)": freq[0][1],
        "Distance-only correct": plain[0][1] == gold,
        "Frequency version correct": freq[0][1] == gold,
        "Top-5 distance-only": ", ".join(w for d,w in plain),
        "Top-5 frequency": ", ".join(w for d,w in freq)
    })
spell_df = pd.DataFrame(rows)
display(spell_df)
print("Top-1 accuracy, distance-only:", round(spell_df["Distance-only correct"].mean(), 3))
print("Top-1 accuracy, frequency tie-break:", round(spell_df["Frequency version correct"].mean(), 3))

**Observations:** Edit distance ranks strings by character edits, not by context or intended meaning. A frequency tie-break can help when several candidates have the same distance, but the CMU pronunciation dictionary is only a rough frequency proxy; a real frequency corpus such as word counts from a large text collection is better. The correct word may not be in the dictionary, and the closest spelling is not always the intended word.

**Why `their`/`there` errors are not caught:** Both are valid words, so a dictionary spell-checker has no reason to flag either. Detecting real-word errors requires context, grammar, or a language model (e.g., “Their going home” versus “They’re going home”).

**Complexity note:** This straightforward suggestion function computes distance to every dictionary entry. For a large vocabulary it can be slow; a BK-tree, trie-based search, or SymSpell-style index can reduce lookup time.

## Submission checklist
- [x] Task 1: mixed-case corpus, vocabulary comparison, ambiguity examples, discussion.
- [x] Task 2: 15 noisy examples, separate regex functions, pipeline, ordering and over-cleaning discussion.
- [x] Task 3: NLTK stop words, before/after metrics, two top-15 plots, custom domain list.
- [x] Task 4: negation, MT examples, chatbot intents, safer stop-word strategy.
- [x] Task 5: 30+ words, Porter/Snowball comparison, over-/under-stemming discussion.
- [x] Task 6: 100+ word paragraph, POS/lemma/stem table, timing.
- [x] Task 7: four configurations on 60 documents, vocabulary/length/accuracy table.
- [x] Task 8: recursive and iterative edit distance, DP tables, backtrace, substitution cost comparison.
- [x] Task 9: 5,000+ word dictionary attempt, top-k suggestions, 15 tests, top-1 accuracy comparison.